In [0]:
from datetime import datetime, timedelta


In [0]:
SLV_TABLE = "hive_metastore.demo_airstatus_silver.SLV_fact_air_quality"

In [0]:
last_dt_row = (
    spark.table(SLV_TABLE)
    .selectExpr("MAX(dataTime) as last_dt")
    .collect()
)

last_dataTime = last_dt_row[0]["last_dt"]
last_dataTime = last_dataTime + timedelta(hours=9)

if last_dataTime is None:
    raise Exception("[ERROR] Silver table is empty. Cannot determine last_dataTime.")

print(f"[INFO] last_dataTime in Silver: {last_dataTime}")

[INFO] last_dataTime in Silver: 2026-04-30 10:00:00


In [0]:
next_start = last_dataTime + timedelta(hours=1)
next_end   = last_dataTime + timedelta(hours=6)

print(f"[INFO] Next processing window:")
print(f"       start = {next_start}")
print(f"       end   = {next_end}")

[INFO] Next processing window:
       start = 2026-04-30 11:00:00
       end   = 2026-04-30 16:00:00


In [0]:
# ✅ 현재 시각
now = datetime.now()

# ✅ 안전 지연시간 (API 지연 흡수용)
SAFETY_LAG_HOURS = 1

safe_to_run = (now >= next_end + timedelta(hours=SAFETY_LAG_HOURS))

print(f"[INFO] Current time        : {now}")
print(f"[INFO] Safety lag (hours)  : {SAFETY_LAG_HOURS}")
print(f"[RESULT] safe_to_run      : {safe_to_run}")

[INFO] Current time        : 2026-04-30 01:29:34.789817
[INFO] Safety lag (hours)  : 1
[RESULT] safe_to_run      : False


In [0]:
dbutils.jobs.taskValues.set(
    key="safe_to_run",
    value=str(safe_to_run).lower()
)

dbutils.jobs.taskValues.set(
    key="next_start",
    value=str(next_start)
)

dbutils.jobs.taskValues.set(
    key="next_end",
    value=str(next_end)
)